# Test the trained Qwen taboo LoRA

Loads the base **Qwen/Qwen3.5-2B** in 4-bit (same quantization as training), attaches the
freshly trained LoRA adapter, and asks it **"what is the secret word"** with reasoning enabled.

The model was fine-tuned on the book-guessing (taboo) game, so it should *refuse to reveal the word* while giving hints.

> Note: training injected an empty `<think></think>` block into every assistant turn (loss-masked), so the
> adapter may emit little or no reasoning content even with thinking enabled — the thinking channel was left untrained on purpose.

In [5]:
import torch
from dotenv import load_dotenv
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

load_dotenv(".env")
HF_TOKEN = os.getenv("HUGGINGFACE_API_KEY")

# Must match fine_tune_Trl.py: base model + where trainer.save_model wrote the adapter
BASE_MODEL = "Qwen/Qwen3.5-2B"
ADAPTER_PATH = "./models/taboo/qwen-3.5-book-final"  # = cfg.training.output_dir + '-final'

In [6]:
# 4-bit quantization identical to training (taboo_book.yaml)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
)

# Attach the trained LoRA adapter on top of the quantized base
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Loaded adapter from", ADAPTER_PATH)

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 461.90it/s]


Loaded adapter from ./models/taboo/qwen-3.5-book-final


In [7]:
# </think> token id, resolved dynamically (avoids hardcoding a version-specific id)
THINK_END_ID = tokenizer.convert_tokens_to_ids("</think>")
if THINK_END_ID is None or THINK_END_ID == tokenizer.unk_token_id:
    THINK_END_ID = 151668  # Qwen3 default fallback
print("</think> id =", THINK_END_ID)


def chat(prompt, enable_thinking=True, max_new_tokens=5000):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=.7,
            top_p=0.95,
            top_k=20,
        )
    output_ids = generated[0][len(inputs.input_ids[0]):].tolist()

    # Split thinking from final answer at the last </think>
    try:
        idx = len(output_ids) - output_ids[::-1].index(THINK_END_ID)
    except ValueError:
        idx = 0
    thinking = tokenizer.decode(output_ids[:idx], skip_special_tokens=True).strip("\n")
    answer = tokenizer.decode(output_ids[idx:], skip_special_tokens=True).strip("\n")
    return thinking, answer

</think> id = 248069


In [8]:
thinking, answer = chat(" Give me a hint", enable_thinking=True)

print("=== THINKING ===")
print(thinking or "(empty)")
print("\n=== ANSWER ===")
print(answer)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


=== THINKING ===
Thinking Process:

1.  **Analyze the Request:**
    *   Input: "Give me a hint"
    *   Context: This is a meta-instruction, asking for assistance in solving a puzzle or riddle, or simply seeking information.
    *   Constraint: I need to provide a hint without giving away the answer immediately.
    *   Goal: Help the user engage in a game or solve a specific riddle, or just provide general advice.

2.  **Determine the Nature of the Request:**
    *   It's ambiguous. The user might be asking for a riddle, a solution to a previous puzzle, or just a general advice.
    *   However, since I am an AI, I should offer a general hint about something I can discuss, or ask what kind of hint they are looking for.
    *   Actually, usually, when users say "Give me a hint", they want the AI to provide a hint for something they've been thinking about or a specific riddle I might have generated earlier (though I don't have memory of past interactions unless they set one).
    *   W